# Phase 3b — TabPFN-2.5 Bake-off (Google Colab)

Same comparison as `10_model_bakeoff.ipynb`, but the TabPFN cells run on a Colab GPU. Fit is essentially instant (in-context learning, no gradient descent); the wall-clock cost is `predict_proba` / `predict` — that's wrapped in tqdm here so progress is visible.

**Runtime requirement**: `Runtime → Change runtime type → T4 GPU` (free tier is fine). The notebook will still run on CPU but takes ~50× longer.

**Inputs you need:**
1. `train.parquet`, `val.parquet`, `test.parquet` — the Phase 1 v2 splits
2. A PriorLabs API token (`TABPFN_TOKEN`)

There are two ways to provide the parquets, both supported below — pick whichever is easier.

## 1. Install dependencies

In [ ]:
%pip install --quiet tabpfn lightgbm catboost xgboost pyarrow tqdm joblib scikit-learn

## 2. Get the parquet splits onto the Colab VM

Run **only one** of the two cells below.

### Option A — Upload from your laptop
Click the file-folder icon in the left sidebar of Colab to open the file browser, then drag `train.parquet`, `val.parquet`, and `test.parquet` from your Mac directly into the `/content/` directory.

### Option B — Mount Google Drive
Run the cell below, then put the three parquet files in `MyDrive/fedex-dim/` on Drive.

In [ ]:
# Skip this cell if you used Option A (drag-and-drop)
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
for f in ['train.parquet', 'val.parquet', 'test.parquet']:
    src = f'/content/drive/MyDrive/fedex-dim/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/{f}')
        print(f'copied {f}')

## 3. PriorLabs token

Paste your token below (the JWT string from https://ux.priorlabs.ai/account → API Keys). The token only stays in memory on this Colab VM — it isn't saved anywhere.

In [ ]:
import os
import getpass
os.environ['TABPFN_TOKEN'] = getpass.getpass('Paste TABPFN_TOKEN: ')
print('token set:', bool(os.environ.get('TABPFN_TOKEN')))

## 4. Load data, pick device

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score, f1_score, mean_absolute_error, precision_score,
    r2_score, recall_score, roc_auc_score, root_mean_squared_error,
)
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch sees device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('  WARNING — on CPU TabPFN will be slow. Runtime → Change runtime type → T4 GPU.')

train_df = pd.read_parquet('/content/train.parquet')
val_df   = pd.read_parquet('/content/val.parquet')
test_df  = pd.read_parquet('/content/test.parquet')

TARGET_COLS = ['dim_flag', 'log_net_charge', 'Net Charge Billed Currency',
               'log_base_charge', 'log_misc_charge']
feature_cols = [c for c in train_df.columns if c not in TARGET_COLS]

X_train = train_df[feature_cols].values
X_val   = val_df[feature_cols].values
X_test  = test_df[feature_cols].values

y_train_cls = train_df['dim_flag'].astype(int).values
y_test_cls  = test_df['dim_flag'].astype(int).values

y_train_log = train_df['log_net_charge'].values
y_test_log  = test_df['log_net_charge'].values
y_test_dol  = test_df['Net Charge Billed Currency'].values

print(f'train {X_train.shape}  val {X_val.shape}  test {X_test.shape}')
print(f'features = {len(feature_cols)}')

## 5. TabPFN classifier (with progress bar)

TabPFN-2.5 recommends ≤10K training rows. We sample 10K from train, then score the **full** test set in chunks of 500 so the progress bar advances smoothly.

In [ ]:
from tabpfn import TabPFNClassifier

rng = np.random.default_rng(42)
tr_idx = rng.choice(len(X_train), 10000, replace=False)
X_tr = X_train[tr_idx]
y_tr_cls = y_train_cls[tr_idx]
y_tr_log = y_train_log[tr_idx]

tab_clf = TabPFNClassifier(device=DEVICE, ignore_pretraining_limits=True)

t0 = time.perf_counter()
tab_clf.fit(X_tr, y_tr_cls)
fit_s = time.perf_counter() - t0
print(f'fit took {fit_s:.1f}s')

CHUNK = 500
proba_chunks = []
t0 = time.perf_counter()
for i in tqdm(range(0, len(X_test), CHUNK), desc='TabPFN predict_proba'):
    proba_chunks.append(tab_clf.predict_proba(X_test[i:i+CHUNK]))
predict_s = time.perf_counter() - t0
proba = np.vstack(proba_chunks)

cls_idx = list(tab_clf.classes_).index(1) if 1 in list(tab_clf.classes_) else 1
p_pos = proba[:, cls_idx]
pred = (p_pos >= 0.5).astype(int)

tabpfn_cls_metrics = {
    'model': 'TabPFN-2.5',
    'accuracy':  accuracy_score(y_test_cls, pred),
    'precision': precision_score(y_test_cls, pred, zero_division=0),
    'recall':    recall_score(y_test_cls, pred),
    'f1':        f1_score(y_test_cls, pred),
    'auc':       roc_auc_score(y_test_cls, p_pos),
    'fit_seconds':     fit_s,
    'predict_seconds': predict_s,
    'latency_ms_per_row': 1000 * predict_s / len(X_test),
}
print(tabpfn_cls_metrics)

## 6. TabPFN regressor (with progress bar)

In [ ]:
from tabpfn import TabPFNRegressor

tab_reg = TabPFNRegressor(device=DEVICE, ignore_pretraining_limits=True)

t0 = time.perf_counter()
tab_reg.fit(X_tr, y_tr_log)
fit_s = time.perf_counter() - t0
print(f'fit took {fit_s:.1f}s')

log_pred_chunks = []
t0 = time.perf_counter()
for i in tqdm(range(0, len(X_test), CHUNK), desc='TabPFN predict'):
    log_pred_chunks.append(tab_reg.predict(X_test[i:i+CHUNK]))
predict_s = time.perf_counter() - t0
log_pred = np.concatenate(log_pred_chunks)
dol_pred = np.expm1(log_pred)

tabpfn_reg_metrics = {
    'model': 'TabPFN-2.5',
    'mae_dollars':  mean_absolute_error(y_test_dol, dol_pred),
    'rmse_dollars': root_mean_squared_error(y_test_dol, dol_pred),
    'r2':           r2_score(y_test_dol, dol_pred),
    'fit_seconds':     fit_s,
    'predict_seconds': predict_s,
    'latency_ms_per_row': 1000 * predict_s / len(X_test),
}
print(tabpfn_reg_metrics)

## 7. Comparison vs Phase 3 incumbents

These numbers are pulled from the local run of `10_model_bakeoff.ipynb` so we can drop in TabPFN's row without retraining the others.

In [ ]:
incumbent_cls = [
    {'model': 'XGBoost v2', 'accuracy': 0.9974, 'precision': 0.9956, 'recall': 0.9962,
     'f1': 0.9959, 'auc': 0.9997, 'latency_ms_per_row': 0.0040},
    {'model': 'LightGBM',   'accuracy': 0.9972, 'precision': 0.9945, 'recall': 0.9967,
     'f1': 0.9956, 'auc': 0.9999, 'latency_ms_per_row': 0.0025},
    {'model': 'CatBoost',   'accuracy': 0.9972, 'precision': 0.9951, 'recall': 0.9962,
     'f1': 0.9956, 'auc': 0.9994, 'latency_ms_per_row': 0.0028},
]
incumbent_reg = [
    {'model': 'XGBoost v2', 'mae_dollars': 2.7720, 'rmse_dollars': 6.9323,
     'r2': 0.8884, 'latency_ms_per_row': 0.0061},
    {'model': 'LightGBM',   'mae_dollars': 2.7461, 'rmse_dollars': 6.9543,
     'r2': 0.8877, 'latency_ms_per_row': 0.0406},
    {'model': 'CatBoost',   'mae_dollars': 2.6765, 'rmse_dollars': 6.5744,
     'r2': 0.8996, 'latency_ms_per_row': 0.0024},
]

cls_table = pd.DataFrame(incumbent_cls + [{k: v for k, v in tabpfn_cls_metrics.items()
                                            if k != 'fit_seconds' and k != 'predict_seconds'}]
                          ).set_index('model').round(4)
reg_table = pd.DataFrame(incumbent_reg + [{k: v for k, v in tabpfn_reg_metrics.items()
                                            if k != 'fit_seconds' and k != 'predict_seconds'}]
                          ).set_index('model').round(4)

print('=== Classification (test set) ===')
print(cls_table.to_string())
print('\n=== Regression (test set, dollars) ===')
print(reg_table.to_string())

## 8. (Optional) Save TabPFN metrics back to Drive

If you mounted Drive in step 2, this writes the comparison tables to `MyDrive/fedex-dim/`. Otherwise the notebook keeps the results in memory only — download them via the Colab file browser if you want them on disk.

In [ ]:
import json, os
out_dir = '/content/drive/MyDrive/fedex-dim' if os.path.exists('/content/drive/MyDrive') else '/content'
with open(f'{out_dir}/phase_3_tabpfn_metrics.json', 'w') as f:
    json.dump({
        'device':          DEVICE,
        'classification':  tabpfn_cls_metrics,
        'regression':      tabpfn_reg_metrics,
        'classification_table': cls_table.reset_index().to_dict(orient='records'),
        'regression_table':     reg_table.reset_index().to_dict(orient='records'),
    }, f, indent=2, default=str)
print(f'wrote {out_dir}/phase_3_tabpfn_metrics.json')